# CIFAR-10 SupCon Augmentations

Visualize the two-view augmentation pipeline used by supervised contrastive training.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import torch
import yaml

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.data.cifar10 import _CIFAR10_MEAN, _CIFAR10_STD, get_cifar10_supcon_loaders

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs" / "cifar10_supcon.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg["data"]["batch_size"] = 8
cfg["data"]["num_workers"] = 0

train_loader, memory_loader, val_loader, train_dataset, val_dataset = get_cifar10_supcon_loaders(
    data_root=cfg["data"]["root"],
    batch_size=cfg["data"]["batch_size"],
    num_workers=cfg["data"]["num_workers"],
    smoke_test=False,
)

In [ ]:
CIFAR10_CLASSES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]


def denormalize(image: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(_CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(_CIFAR10_STD).view(3, 1, 1)
    return (image.cpu() * std + mean).clamp(0.0, 1.0)


def show_augmented_batch(num_examples: int = 6) -> None:
    views, labels = next(iter(train_loader))
    num_examples = min(num_examples, views.size(0))

    fig, axes = plt.subplots(num_examples, 2, figsize=(6, 3 * num_examples))
    if num_examples == 1:
        axes = [axes]

    for row in range(num_examples):
        class_name = CIFAR10_CLASSES[int(labels[row])]
        for col in range(2):
            ax = axes[row][col]
            image = denormalize(views[row, col]).permute(1, 2, 0)
            ax.imshow(image)
            ax.set_title(f"{class_name} | view {col + 1}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_augmented_batch(num_examples=6)